#**Week 5 Day 4**

1. Scikit-learn Pipeline: chain imputer → scaler → encoder → model in one object
2. ColumnTransformer: different transforms for numerical vs categorical columns
3.	Custom transformer: subclass BaseEstimator + TransformerMixin — wrap your log-transform in it



In [46]:
import pandas as pd
import numpy as np

In [47]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Roadmap/Datasets/titanic/train.csv")

df['Sex_encoded'] = df['Sex'].map({'male':0, 'female': 1})
features = ['Pclass', 'Age', 'Fare', 'Sex_encoded']

X_numeric = df[features]
y = df['Survived']

In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
 12  Sex_encoded  891 non-null    int64  
dtypes: float64(2), int64(6), object(5)
memory usage: 90.6+ KB


#### Task 1
1. Basic Pipeline Chain SimpleImputer → StandardScaler → RandomForestClassifier on Titanic.
2.  One .fit() call, one .predict() call.


In [48]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

X_numeric_train, X_numeric_test, y_train, y_test = train_test_split(X_numeric, y, test_size = 0.3, random_state = 42)

In [49]:
#pipeline

pipeline = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier())
])

# step 1 pipeline.fit
pipeline.fit(X_numeric_train,y_train)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', RandomForestClassifier())])

In [50]:
#Step 2: Pipeline.predict

from sklearn.metrics import accuracy_score
y_pred = pipeline.predict(X_numeric_test)
print(f"PipeLine Accuracy Score {accuracy_score(y_test,y_pred):.3f}")

PipeLine Accuracy Score 0.806


#### pipeline.fit(X_train, y_train):
 1.  Step 1: imputer.fit_transform(X_train)      → fills nulls
  2. Step 2: scaler.fit_transform(imputed_data)  → scales
  3. Step 3: model.fit(scaled_data, y_train)     → trains

#### pipeline.predict(X_test):
  1. Step 1: imputer.transform(X_test)           → fit already done, just transform
  2. Step 2: scaler.transform(imputed_test)      → same
  3. Step 3: model.predict(scaled_test)          → predicts

#### Task 2 - ColumnTransformer
1. Numerical columns get imputer + scaler.
2.  Categorical columns get imputer + OneHotEncoder.
- Combine into one transformer.



##### Titanic dataset
1. Numeric-> Pclass, Age, Fare : need Imputer + Scaler
2. Categorical -> Sex_encoded : need Imputer + Encoder

In [51]:
X = df[['Pclass', 'Age', 'Fare', 'Sex']]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 42)

In [52]:
X

,Pclass,Age,Fare,Sex
0,3,22.0,7.2500,male
1,1,38.0,71.2833,female
2,3,26.0,7.9250,female
3,1,35.0,53.1000,female
4,3,35.0,8.0500,male
...,...,...,...,...
886,2,27.0,13.0000,male
887,1,19.0,30.0000,female
888,3,NaN,23.4500,female
889,1,26.0,30.0000,male


In [53]:
from sklearn.compose import ColumnTransformer

numerical_features = ['Pclass', 'Age', 'Fare']
categorical_features = ['Sex']


#Transformer
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy= 'median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder())
])


#preprocessor
preprocessor = ColumnTransformer([
    ('num', numerical_transformer, numerical_features),
    ('cat', categorical_transformer, categorical_features)
])

In [54]:
#Pipeline : preprocessor + model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state= 42))
])

full_pipeline.fit(X_train, y_train)
y_pred = full_pipeline.predict(X_test)

print(f"Column Transform Pipline accuracy: {accuracy_score(y_test, y_pred):.3f}")

Column Transform Pipline accuracy: 0.795


#### Task 3 - Custom Transformer
1.  Subclass BaseEstimator + TransformerMixin to wrap       np.log1p
2. Plug it into a Pipeline.


In [55]:
from typing import Self
from sklearn.base import BaseEstimator, TransformerMixin

class LogTransformer(BaseEstimator, TransformerMixin):
  def fit(self, X,y =None):
    return self

  def transform(self, X):
    return np.log1p(X.select_dtypes(include='number'))


In [66]:
log_pipeline = Pipeline([
    ('log', LogTransformer()),
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state= 42))
])

In [67]:
log_pipeline.fit(X_train, y_train)

Pipeline(steps=[('log', LogTransformer()), ('scaler', StandardScaler()),
                ('model', RandomForestClassifier(random_state=42))])

In [69]:
y_pred = log_pipeline.predict(X_test)

print(f"Log Transform Pipeline accuracy: {accuracy_score(y_test, y_pred)}")

Log Transform Pipeline accuracy: 0.7014925373134329


#### Task 4

## Why Use Pipeline Over Manual Steps?
1. Manual steps risk data leakage : if you fit a scaler on the full dataset
before splitting, test data influences the scaler.
2. Pipeline fits transformers on training data only, transforms test data automatically.
3.  One fit() call, one predict() call. Works directly with GridSearchCV.

### Pipeline
1. Chains steps sequentially: each step's output becomes next step's input.
- fit() → fit_transform() each step in order, fit() on final model.
- predict() → transform() each step, predict() on final model.


### ColumnTransformer
1. Applies different transformations to different column subsets simultaneously.
- Numerical → imputer + scaler.
- Categorical → imputer + OneHotEncoder.
2. Combines outputs into one feature matrix automatically.

### Custom Transformer
1. Subclass BaseEstimator + TransformerMixin.
2. Write only fit() and transform() :get fit_transform(), get_params(),
set_params() automatically.
3. Makes any custom logic plug directly into Pipeline.
- fit() returns self when no parameters to learn from data.

### Key Rule
1. Always use Pipeline in production: never fit transformers manually
on full dataset.
2. Apply custom transformers selectively via ColumnTransformer, not to all columns.